# Lumen 1.3 SFT — Qwen3-8B QLoRA (Unsloth)

Trains Lumen on the Axion training mix produced by `gen-trajectories.mjs` + `reformat-hf.mjs`.

**Workflow (units are precious):**
1. Upload `training/data/*.jsonl` to Google Drive folder `lumen-data/` (or Kaggle dataset `lumen-data`).
2. Run this notebook END-TO-END with `TINY = True` on a **free GPU** (Kaggle T4 / Colab free T4) until it finishes without errors.
3. Only then: Colab **A100**, `TINY = False`. Expect ~3–3.5 h for one epoch on ~20k samples (~60M tokens).

Checkpoints go to Drive every `SAVE_STEPS` steps — a disconnect loses minutes, not the run.
Resume by re-running with `RESUME = True`.

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
TINY        = True          # True = 200-sample dry run (debug on free GPU first!)
RESUME      = False         # resume from last checkpoint in OUTPUT_DIR
BASE_MODEL  = 'unsloth/Qwen3-8B-unsloth-bnb-4bit'
MAX_SEQ     = 8192          # A100 40GB handles 8192 comfortably with QLoRA
LORA_R      = 32
LORA_ALPHA  = 64
LR          = 2e-4
EPOCHS      = 1
BATCH       = 2             # per-device
GRAD_ACCUM  = 16            # effective batch 32
SAVE_STEPS  = 200
SEED        = 3407

# Mix ratios by slice (task field -> target share). Slices that run out are
# used fully; shares renormalize over what exists.
MIX = {
    'axion-native': 0.15,   # trajectories.jsonl (task field varies -> grouped below)
    'swe-agentic':  0.30,   # swe-smith.jsonl
    'code-instruct':0.20,   # magicoder.jsonl
    'general':      0.25,   # tulu.jsonl
    'tool-calling': 0.10,   # hermes.jsonl
}
TARGET_TOTAL = 200 if TINY else 20_000

In [ ]:
# ── Install ────────────────────────────────────────────────────────────────
%pip install -q unsloth
import torch
print('GPU:', torch.cuda.get_device_name(0), '| VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9), 'GB')

In [ ]:
# ── Locate data (Colab Drive or Kaggle input) ──────────────────────────────
import os, glob
if os.path.exists('/kaggle'):
    DATA_DIR   = '/kaggle/input/lumen-data'          # add the dataset to the notebook
    OUTPUT_DIR = '/kaggle/working/lumen-ckpt'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR   = '/content/drive/MyDrive/lumen-data'
    OUTPUT_DIR = '/content/drive/MyDrive/lumen-ckpt'  # checkpoints survive disconnects
files = sorted(glob.glob(f'{DATA_DIR}/*.jsonl'))
assert files, f'No .jsonl files found in {DATA_DIR} — upload training/data/*.jsonl there.'
print('\n'.join(files))

In [ ]:
# ── Load slices, build the mix ─────────────────────────────────────────────
import json, random
random.seed(SEED)

SLICE_OF_FILE = {  # filename stem -> mix key
    'trajectories': 'axion-native', 'swe-smith': 'swe-agentic',
    'magicoder': 'code-instruct', 'tulu': 'general', 'hermes': 'tool-calling',
}
slices = {k: [] for k in MIX}
for f in files:
    stem = os.path.basename(f).rsplit('.', 1)[0]
    key = SLICE_OF_FILE.get(stem)
    if key is None:
        print('skipping unknown file', f); continue
    seen_ids = set()
    for line in open(f, encoding='utf-8'):
        try: rec = json.loads(line)
        except: continue
        if rec.get('id') in seen_ids: continue   # batches append -> dedupe
        seen_ids.add(rec.get('id'))
        if rec.get('messages'): slices[key].append(rec)

avail = {k: len(v) for k, v in slices.items()}
print('available:', avail)
norm = sum(share for k, share in MIX.items() if avail.get(k))
mix = []
for k, share in MIX.items():
    want = int(TARGET_TOTAL * share / norm)
    take = random.sample(slices[k], min(want, len(slices[k]))) if slices[k] else []
    mix.extend(take)
    print(f'{k}: want {want}, took {len(take)}')
random.shuffle(mix)
print('total mix:', len(mix))

In [ ]:
# ── Model + tokenizer ──────────────────────────────────────────────────────
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ, load_in_4bit=True, dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none', use_gradient_checkpointing='unsloth', random_state=SEED,
)

In [ ]:
# ── Render with Qwen3 chat template (tools included) ───────────────────────
# Our schema is already OpenAI-style messages + tools; Qwen3's template
# renders tools into the system region and tool_calls as <tool_call> JSON.
from datasets import Dataset

def render(rec):
    try:
        text = tokenizer.apply_chat_template(
            rec['messages'], tools=rec.get('tools') or None,
            tokenize=False, add_generation_prompt=False,
        )
        return {'text': text}
    except Exception as e:
        return {'text': ''}

ds = Dataset.from_list(mix).map(render, remove_columns=None)
before = len(ds)
ds = ds.filter(lambda r: len(r['text']) > 0)
# Drop samples that exceed MAX_SEQ after tokenization (truncating a trajectory
# mid-tool-call teaches garbage). Report what we lose.
def short_enough(r):
    return len(tokenizer(r['text'], add_special_tokens=False).input_ids) <= MAX_SEQ
ds = ds.filter(short_enough, num_proc=2)
print(f'render failures: {before - len(ds)} dropped (template errors + over {MAX_SEQ} tokens)')
print('final training samples:', len(ds))
print('\n--- sample render (first 1200 chars) ---\n', ds[0]['text'][:1200])

In [ ]:
# ── Train (loss on assistant tokens only) ──────────────────────────────────
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    dataset_text_field='text',
    args=SFTConfig(
        per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type='cosine',
        warmup_ratio=0.03, logging_steps=5, save_steps=SAVE_STEPS,
        save_total_limit=2, output_dir=OUTPUT_DIR, seed=SEED,
        max_seq_length=MAX_SEQ, packing=False,
        bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(),
        optim='adamw_8bit', report_to='none',
    ),
)
# Mask everything except assistant responses (incl. their <tool_call> blocks).
trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)
stats = trainer.train(resume_from_checkpoint=RESUME)
print(stats)

In [ ]:
# ── Save adapters (small, always) + merged 16-bit (for vLLM serving) ───────
FINAL = f'{OUTPUT_DIR}/lumen-1.3-lora'
model.save_pretrained(FINAL); tokenizer.save_pretrained(FINAL)
print('LoRA adapter saved to', FINAL)
if not TINY:
    # ~16GB write — needs Drive space; this is what the lumen endpoint serves.
    model.save_pretrained_merged(f'{OUTPUT_DIR}/lumen-1.3-merged', tokenizer, save_method='merged_16bit')
    print('merged model saved')

In [ ]:
# ── Sanity check: does it call Axion tools? ────────────────────────────────
import json as _json
FastLanguageModel.for_inference(model)
axion_sample = next(r for r in mix if r['task'] not in ('general', 'tool-calling'))
sys_msg = axion_sample['messages'][0]      # Axion system prompt from real data
tools   = axion_sample.get('tools')
probe = [sys_msg, {'role': 'user', 'content': 'The test in test.js is failing. Find the bug in lib.js and fix it.'}]
inputs = tokenizer.apply_chat_template(probe, tools=tools, tokenize=True,
                                       add_generation_prompt=True, return_tensors='pt').to('cuda')
out = model.generate(input_ids=inputs, max_new_tokens=300, temperature=0.3, do_sample=True)
text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=False)
print(text)
print('\nPASS: emits a tool call' if '<tool_call>' in text else '\nCHECK: no tool call emitted — inspect output above')

## After training

- **Serve**: load `lumen-1.3-merged` in vLLM with `--enable-auto-tool-choice --tool-call-parser hermes`
  behind the existing lumen endpoint (`api.amplifiedsmp.org/v1`). Axion needs zero changes —
  `/model lumen` already points there.
- **Eval**: back on your machine, run held-out seeds through the generator against the deployed model:
  `node training/gen-trajectories.mjs --n 50 --seed 999000 --model lumen` and compare pass-rate
  against the same seeds with the base Qwen3-8B served the same way.
- Also spot-check general chat (a few plain questions) to confirm no catastrophic forgetting.